In [ ]:
import plotly.graph_objects as pgo
import plotly.subplots as psu

from waffles.input.raw_ROOT_reader import WaveformSet_from_ROOT_file

from my_files.ProtoDUNE_HD_APA_maps import APA_map
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#nome do arquivo
file="my_files/runs.txt"
# Abre o arquivo em modo de leitura
with open(file, 'r') as arquivo:
    # Lê todas as linhas do arquivo e armazena em uma lista
    linhas = arquivo.readlines()

# Remove os caracteres de nova linha (\n) de cada linha
linhas = [linha.strip() for linha in linhas]

In [ ]:
#faz uma lista das runs usadas
from my_files.useful_functions import extract_run_number

n_run=[]
n_run=[extract_run_number(files_names) for files_names in linhas if extract_run_number(files_names)]
n_run_list=list(set(n_run))
n_run_list.sort()
n_run_list

n_files=len(linhas)
n_run=len(n_run_list)
n_run_list

In [ ]:
#retornar posicoes da matriz para um determinado canal e APA
def return_index(channel, endpoint):
    for k in range(1,5):    
        for i in range (10):
            for j in range(4):
                if (APA_map[k].Data[i][j].Endpoint == endpoint and APA_map[k].Data[i][j].Channel == channel):
                    return k,i,j
    return -1,-1,-1

In [ ]:
wfset = [WaveformSet_from_ROOT_file( filepath,
                                            bulk_data_tree_name = 'raw_waveforms', 
                                            meta_data_tree_name = 'metadata',
                                            set_offset_wrt_daq_window = False,
                                            read_full_streaming_data = False,
                                            truncate_wfs_to_minimum = False,
                                            start_fraction = 0.0,
                                            stop_fraction = 1.0,
                                            library = 'pyroot',
                                            subsample = 1) for filepath in linhas]

In [ ]:
#pega o tamanho das variavies: quantas wfs poor file
samples=[len(w.Waveforms) for w in wfset]
samples

In [ ]:
set_of_endpoints = wfset[0].get_set_of_endpoints()
list_endpoints=list(set_of_endpoints)
list_endpoints.sort()
list_endpoints

In [ ]:
#normalizar pelo tempo gasto, para ser justo
time_list = [[[] for _ in range(len(list_endpoints))] for _ in range(n_run)]
list_aux = [[[] for _ in range(len(list_endpoints))] for _ in range(n_run)]
for n in range(n_files):
    this_index=n_run_list.index(extract_run_number(linhas[n]))

    for aux in range(len(wfset[n].Waveforms)):
        index = list_endpoints.index(wfset[n].Waveforms[aux].Endpoint)
        list_aux[this_index][index].append(wfset[n].Waveforms[aux].Timestamp)

for n in range(n_files):
    for k in range(len(list_endpoints)):
        time_list[n][k].append(np.max(list_aux[n][k]))
        time_list[n][k].append(np.min(list_aux[n][k]))
        list_aux[n][k].clear()
        list_aux[n][k]=[]

In [ ]:
#para acelerar o processo, vamos so olhar onde queremos --> isso faz ganhar muito tempo aqui
#lembrando que nesse caso aqui, vou so trabalhar com channel 21, endpoint 112, APA4 --> (k,i,j):(3,6,0)
ch_index=21
endpoint_index=112


#preparando tudo
data_wf=[[[[{"wf": [], "wf_0": [], "wf_dec": [], "integral": [], "amplitude": [], "baseline": [], "wf_avg": None, 
             "fast_integral": [], "noise": [], "fprompt": [], "time_0": [], "time_start": [], "integral_0":[], "integral_all":[], "endpoint": None } 
             for _ in range(4)] for _ in range (10)] for _ in range(4)] for _ in range(n_run)]


for n in range(0,n_files):
    #estamos olhando uma run e precisamos varrer as waveforms
    this_index=n_run_list.index(extract_run_number(linhas[n]))
    for i in range(0,samples[n]):
        ch=wfset[n].Waveforms[i].Channel
        end=wfset[n].Waveforms[i].Endpoint
        
        if (ch==ch_index) and (end==endpoint_index):
             
            table,row,col= return_index(ch,end)
            if (table !=-1 and row!=-1 and col!=-1):
                data_wf[this_index][table-1][row][col]["wf"].append(wfset[n].Waveforms[i].Adcs)
        

In [ ]:
#carregar template
n_zeros=300

#spe_wf = np.loadtxt('spe_APA3_ch21_Endpoint111_y_x6_0.txt')
spe_wf = np.loadtxt('hpk_PD1.txt')
spe_wf_0 = np.concatenate([spe_wf,np.zeros(n_zeros)])*(-1)
#fft
spe_fft=np.fft.fft(spe_wf_0)

In [ ]:
plt.plot(spe_wf_0)
plt.axhline(0,linestyle="--",color="red")

In [ ]:
#lembrando que nesse caso aqui, vou so trabalhar com channel 21, endpoint 111, APA4 --> (k,i,j):(3,6,0)

#removendo baseline e fazendo zeropadding
baseline_index=40
k=3
i=6
j=0

window_size = 30
kernel = np.ones(window_size) / window_size

for n in range(n_run):
    if data_wf[n][k][i][j]["wf"] != []: 
        tam=len(data_wf[n][k][i][j]["wf"])
        data_wf[n][k][i][j]["baseline"]=[]

        for aux in range(tam):
            data_wf[n][k][i][j]["baseline"].append(np.mean(data_wf[n][k][i][j]["wf"][aux][0:baseline_index],
                                                                           ))
            #data_wf[n][k][i][j]["wf"][aux]= data_wf[n][k][i][j]["wf"][aux]-data_wf[n][k][i][j]["baseline"][aux]
            data_wf[n][k][i][j]["wf_0"].append(  np.concatenate([np.zeros(n_zeros),data_wf[n][k][i][j]["wf"][aux]-data_wf[n][k][i][j]["baseline"][aux]]))
            data_wf[n][k][i][j]["wf"][aux]= data_wf[n][k][i][j]["wf"][aux]-data_wf[n][k][i][j]["baseline"][aux]
            #data_wf[n][k][i][j]["wf"][aux]=  np.convolve(data_wf[n][k][i][j]["wf"][aux],kernel,"same")

In [ ]:
sigma=8
mean=700
xx=np.arange(len(data_wf[0][k][i][j]["wf_0"][500]))
g=1/np.sqrt(2*np.pi*sigma*sigma)*np.exp(-np.pow(xx-mean,2)/(2*sigma*sigma))

this_wf=data_wf[0][k][i][j]["wf_0"][500]

yy=np.convolve(this_wf,g)[700:len(this_wf)+700]

plt.plot(this_wf)
plt.plot(yy)

In [ ]:
""" #avg wf for wiener filter
s_avg=[np.zeros(len(data_wf[0][k][i][j]["wf"][0])) for _ in range(n_files)]
tam_wf=len(data_wf[0][k][i][j]["wf"][0])

for n in range(n_files):
    n_wf = len(data_wf[n][k][i][j]["wf"])

    for aux in range(tam_wf):
        for index_wf in range(n_wf):
            
            s_avg[n][aux]= s_avg[n][aux]+data_wf[n][k][i][j]["wf"][index_wf][aux]
             
    s_avg[n]=s_avg[n]/n_wf     """

In [ ]:
tam_s=1024
x=np.arange(tam_s)
slow=1.6e-6/16e-9
fast=8e-9/16e-9
s_avg_teor=3*np.exp(-x/slow)+np.exp(-x/fast)
s_avg_teor=40*s_avg_teor/(np.max(s_avg_teor))
s_avg_teor=np.concatenate([np.zeros(n_zeros),s_avg_teor])

In [ ]:
plt.plot(np.abs(np.fft.fft(s_avg_teor)))

spe_fft_abs=np.abs(spe_fft)

plt.yscale("log")

In [ ]:
#o ruido
#vamos preparar o ruido

start=3000
until=8000
noise_wf=[[] for _ in range(n_run)]
tam=len(data_wf[0][k][i][j]["wf_0"][0])-n_zeros


for n in range(n_run):
    size=len(data_wf[n][k][i][j]["wf_0"])
    for index in range(size-3000,size):
        noise_wf[n]=np.concatenate([noise_wf[n],(data_wf[n][k][i][j]["wf_0"][index][n_zeros::])[0:50],[]])
    print(len(noise_wf[n]))
    tam_noise=len(noise_wf[n])
    tam_justo=tam_noise/tam
    noise_wf[n]=noise_wf[n]/tam_justo


In [ ]:
noise_wf_fft_abs=[[] for _ in range(n_run)]


for n in range(n_run):
    indices = np.linspace(0, len(noise_wf[n]) - 1, len(data_wf[n][k][i][j]["wf_0"][0]), dtype=int)
    #print(len(noise_wf_fft_abs[n]))
    noise_wf_fft_abs[n]=np.abs(np.fft.fft(noise_wf[n]))[indices]
    #print(len(noise_wf_fft_abs[n]))
    aux=(noise_wf_fft_abs[n])[1:int(len(noise_wf_fft_abs[n])/2)]
    #print(len(aux))
    noise_wf_fft_abs[n]=noise_wf_fft_abs[n][0:int(len(noise_wf_fft_abs[n])/2)+1]
    #print(len(noise_wf_fft_abs[n]))

    noise_wf_fft_abs[n]=np.concatenate([noise_wf_fft_abs[n],aux[::-1]])
    #print(len(noise_wf_fft_abs[n]))
    noise_wf_fft_abs[n]=noise_wf_fft_abs[n]

In [ ]:
def exp(x,A,mean,std):
    return A*np.exp(-np.pow(x-mean,2)/std)

def exp2(x,A,B,tau):
    return A*np.exp(-x/tau)+B

y_fit=noise_wf_fft_abs[0]
plt.plot(noise_wf_fft_abs[2])
#plt.yscale("log")

from scipy.optimize import curve_fit


x_fit=np.arange(len(y_fit)/2)
params, params_covariance = curve_fit(exp2,x_fit[1:500],np.log(y_fit[1:500]),maxfev=500000)
final_noise=np.concatenate([exp2(x_fit,*params),exp2(x_fit,*params)[::-1]])
plt.plot(np.exp(final_noise))
plt.yscale("log")

final_noise=np.exp(final_noise)



In [ ]:
spe_fft_abs=spe_fft*spe_fft.conjugate()
noise_fft_abs=final_noise*final_noise
s_fft=np.fft.fft(s_avg_teor)


for n in range(n_run):
    if data_wf[n][k][i][j]["wf_0"] != []: 
        tam = len(data_wf[n][k][i][j]["wf_0"])
        
        s_squared=s_fft*s_fft.conjugate()
        #noise_fft_abs=noise_wf_fft_abs[n]*noise_wf_fft_abs[n]
        data_wf[n][k][i][j]["wf_dec"]=[]
        for aux in range(tam):
            
            yy=data_wf[n][k][i][j]["wf_0"][aux]
            yy=np.convolve(yy,g)[700:700+len(yy)]

            aux_fft=np.fft.fft(yy)
            #wiener
            aux_fft=aux_fft*spe_fft.conjugate()
            aux_fft=aux_fft/(spe_fft_abs+noise_fft_abs/(s_squared))
            #normal
            #aux_fft=aux_fft/spe_fft
            data_wf[n][k][i][j]["wf_dec"].append(np.fft.ifft(aux_fft).real)
            data_wf[n][k][i][j]["wf_dec"][aux]=data_wf[n][k][i][j]["wf_dec"][aux]-np.mean(data_wf[n][k][i][j]["wf_dec"][aux][0:200])

In [ ]:
#error analisis (variaçao da amplitude do spe) #nao eh relevante
#erro ao variar std ada convolucao gaussiana # nao eh tao relavante


sigma=1
mean=700
xx=np.arange(len(data_wf[0][k][i][j]["wf_0"][500]))
g=1/np.sqrt(2*np.pi*sigma*sigma)*np.exp(-np.pow(xx-mean,2)/(2*sigma*sigma))



spe_fft_abs=spe_fft*spe_fft.conjugate()
noise_fft_abs=final_noise*final_noise
s_fft=np.fft.fft(s_avg_teor)
error_q=[[] for _ in range(n_run)]


for n in range(n_run):
    if data_wf[n][k][i][j]["wf_0"] != []: 
        tam = len(data_wf[n][k][i][j]["wf_0"])
       
        s_squared=s_fft*s_fft.conjugate()
        #noise_fft_abs=noise_wf_fft_abs[n]*noise_wf_fft_abs[n]
        
        for aux in range(tam):
            var_a=1.25#(A+dA)/A

            yy=var_a*data_wf[n][k][i][j]["wf_0"][aux]
            yy=np.convolve(yy,g)[700:700+len(yy)]

            aux_fft1=np.fft.fft(yy)
            #wiener
            aux_fft1=aux_fft1*spe_fft.conjugate()
            
            aux_fft2=aux_fft1/(spe_fft_abs+noise_fft_abs/(s_squared))

            aux2=np.fft.ifft(aux_fft2).real
            aux2=aux2-np.mean(aux2[0:200])
            
            error_q[n].append(aux2)

In [ ]:
n=5
my_index=1756
#plt.plot(data_wf[n][k][i][j]["wf_dec"][my_index])

from scipy.signal import savgol_filter
window_size = 25
poly_order=3
kernel = np.ones(window_size) / window_size
my_this=data_wf[n][k][i][j]["wf_dec"][my_index]
#my_this=np.convolve(data_wf[n][k][i][j]["wf_dec"][my_index],kernel,mode="same")
#my_this = savgol_filter(data_wf[n][k][i][j]["wf_dec"][my_index],window_size,poly_order)
plt.plot(my_this,".")
plt.axhline(y=0,linestyle="--",color="red")


In [ ]:
""" def sine_func(x, A, omega, phi,B):
    return A * np.sin(omega * x + phi)+B

interval=[[0,100],[900,1300]]

fit_light=[[] for _ in range(n_run)]

from scipy.optimize import curve_fit

for n in range(n_run):
    if data_wf[n][k][i][j]["wf_dec"] != []:
        tam = len(data_wf[n][k][i][j]["wf_dec"])
        for aux in range(tam):
            
            # Separar os dados nos intervalos especificados
            x1 = np.arange(interval[0][0], interval[0][1])        
            x2 = np.arange(interval[1][0], interval[1][1])     
            y_fit=data_wf[n][k][i][j]["wf_dec"][aux]
            y1 = y_fit[interval[0][0]: interval[0][1]]               
            y2 = y_fit[interval[1][0]: interval[1][1]]           

            # Concatenar os índices e os dados para ajuste
            xdata = np.concatenate((x1, x2))
            ydata = np.concatenate((y1, y2))
            try:
                params, params_covariance = curve_fit(sine_func,xdata, ydata, p0=[1, 2*np.pi/1400, np.pi/2,0])
                data_wf[n][k][i][j]["wf_dec"][aux]=y_fit-sine_func(np.arange(0,len(y_fit)), *params)
            except:
                data_wf[n][k][i][j]["wf_dec"][aux]=data_wf[n][k][i][j]["wf_dec"][aux] """

In [ ]:
#vamos calcular as integrais das waveforms deconvoluidas
from scipy.signal import savgol_filter
window_size = 10
poly_order=2
kernel = np.ones(window_size) / window_size

for n in range(n_run):
     if data_wf[n][k][i][j]["wf_dec"] != []:

        data_wf[n][k][i][j]["integral"]=[]
        data_wf[n][k][i][j]["fast_integral"]=[]
        data_wf[n][k][i][j]["fprompt"]=[]
        data_wf[n][k][i][j]["amplitude"]=[]

        tam = len(data_wf[n][k][i][j]["wf_dec"])
        for aux in range(tam):
            #
            my_aux=data_wf[n][k][i][j]["wf_dec"][aux]
            #new_baseline= 0
            data_wf[n][k][i][j]["integral"].append(np.sum(my_aux[250:750]))
            data_wf[n][k][i][j]["amplitude"].append(np.max(my_aux[250:750]))
            data_wf[n][k][i][j]["fast_integral"].append(np.sum(my_aux[250:260]))
            data_wf[n][k][i][j]["fprompt"].append(np.divide(data_wf[n][k][i][j]["fast_integral"][aux],data_wf[n][k][i][j]["integral"][aux]))

In [ ]:
#vamos estimar os erros
#primeiro levando em conta a varaicao de amplitude do singlephotoelectron

k=3
i=6
j=0
charge_error_a=[[] for _ in range(n_run)]

for n in range(n_run):
     tam = len(data_wf[n][k][i][j]["wf_dec"])
     for aux in range(tam):
          charge_error_a[n].append(np.sum(error_q[n][aux][150:1000]))


In [ ]:
""" #preparando tudo
data_wf_selected=[[[[{"wf_dec": [], "integral": [] } 
             for _ in range(4)] for _ in range (10)] for _ in range(4)] for _ in range(n_run)]


f_prompt_range=[0.4,0.6]
integral_range=[250,2000]

for n in range(n_run):
    if data_wf[n][k][i][j]["wf_dec"] != []: 
        tam=len(data_wf[n][k][i][j]["wf_dec"])
        data_wf_selected[n][k][i][j]["wf_dec"]=[]

        for aux in range(tam):
            if data_wf[n][k][i][j]["integral"][aux]>=integral_range[0] and data_wf[n][k][i][j]["integral"][aux]<=integral_range[1]: 
                if data_wf[n][k][i][j]["fprompt"][aux]>=f_prompt_range[0] and data_wf[n][k][i][j]["fprompt"][aux]<=f_prompt_range[1]:
                    data_wf_selected[n][k][i][j]["wf_dec"].append(data_wf[n][k][i][j]["wf_dec"][aux])
 """

In [ ]:
""" #avg wf for light
scint_avg=[np.zeros(len(data_wf[0][k][i][j]["wf_dec"][0])) for _ in range(n_run) ]
tam_wf=len(data_wf[0][k][i][j]["wf_dec"][0])

for n in range(n_run):
    n_wfs=len(data_wf_selected[n][k][i][j]["wf_dec"])
    for aux in range(tam_wf):
        for index_wf in range(n_wfs):

            scint_avg[n][aux]= scint_avg[n][aux]+data_wf_selected[n][k][i][j]["wf_dec"][index_wf][aux]

    scint_avg[n]=scint_avg[n][200:1233]/n_wfs 

for n in range(n_run):
    scint_avg[n]=scint_avg[n]-np.mean(np.concatenate([scint_avg[n][0:50],scint_avg[n][600:1000]]))
 """

In [ ]:
integral_array=np.zeros(n_run)

for n in range(n_run):
    integral_array[n]=np.sum(data_wf[n][k][i][j]["integral"])

In [ ]:
integral_array_error=np.zeros(n_run)

for n in range(n_run):
    integral_array_error[n]=np.sum(charge_error_a[n])

In [ ]:
#preparar o y axis para plotar
y=np.zeros(n_run)
for n in range(n_run):
    delta_t=time_list[n][1][0]-time_list[n][1][1]
    y[n]=integral_array[n]/delta_t#np.sum(data_wf_selected[n][k][i][j]["integral"])#/delta_t

y=y/y[0]
x=[0,20,40,60,80,100,120,140,160,180]

In [ ]:
y

In [ ]:
for n in range(n_run):
    delta_t=time_list[n][1][0]-time_list[n][1][1]
    integral_array_error[n]=integral_array_error[n]/delta_t#np.sum(data_wf_selected[n][k][i][j]["integral"])#/delta_t

integral_array_error=integral_array_error/integral_array_error[0]
print(integral_array_error)

integral_array_error=(integral_array_error-y)

In [ ]:
print(y)

print(integral_array_error)

In [ ]:
#vamos tentar fitar a curva acima
def birks_law(x,k):
    y=np.zeros(len(x))
    if x[0]==0:
        y[0]=1
        y[1::]=(1-1/(1+k/x[1::]))
    else:
        y=1
    return y/y[0]

from scipy.optimize import curve_fit
params, params_covariance = curve_fit(birks_law,np.divide(x,360),y,maxfev=20000,p0=[0.1])
print(*params)
print(*params_covariance)

In [ ]:
plt.errorbar(np.divide(x,360),y,fmt='.',color="black",label="real data",yerr=np.abs(integral_array_error))

plt.xlabel("Efield[kV/cm]")
plt.ylabel("scintillation")
fit_errors = np.sqrt(np.diag(params_covariance))

eixo_x=np.divide(x,360)
var_k=params_covariance[0]
erro_y=np.abs((birks_law(eixo_x,params[0])-birks_law(eixo_x,params[0])))
print(erro_y)
x2=np.linspace(0,0.5,20)

y_fit=birks_law(eixo_x,*params)
plt.errorbar(eixo_x,y_fit,fmt='x',color="red",yerr=erro_y)

plt.plot(x2,birks_law(x2,*params),color="red",label=f"fitted data: 1-1/(1+{params[0]:.2}/E) ")
plt.legend()

In [ ]:

#erro_y=np.abs((birks_law(x2,params[0]+var_k)-birks_law(x2,params[0]))*fit_errors[0]/var_k)

y_fit=birks_law(x2,*params)
plt.errorbar(x2,y_fit,fmt='x',color="red",)


In [ ]:
#avg wf for light
scint_avg=[np.zeros(len(data_wf[0][k][i][j]["wf_dec"][0])) for _ in range(n_files)]
tam_wf=len(data_wf[0][k][i][j]["wf_dec"][0])

f_prompt_range=[0.025,0.075]
integral_range=[200,2000]

cont=0
for n in range(n_files):
    n_wf = len(data_wf[n][k][i][j]["wf_dec"])
    cont=0
    for aux in range(tam_wf):
        for index_wf in range(n_wf):
            if data_wf[n][k][i][j]["integral"][index_wf]<=integral_range[1] and data_wf[n][k][i][j]["integral"][index_wf]>=integral_range[0]:
                if data_wf[n][k][i][j]["fprompt"][index_wf]<=f_prompt_range[1] and data_wf[n][k][i][j]["integral"][index_wf]>=f_prompt_range[0]:
                    scint_avg[n][aux]= scint_avg[n][aux]+data_wf[n][k][i][j]["wf_dec"][index_wf][aux]
                    if aux==0:
                        cont=cont+1
    scint_avg[n]=scint_avg[n]/cont      

In [ ]:
from scipy.optimize import least_squares

def exp_double_func(x, A1, A2,tau1,tau2):
    return A1 * np.exp(-x/tau1) + A2*np.exp(-x/tau2)

def exp_simple_func(x,A,tau):
    return A * np.exp(-x/tau)

def lin_simple_func(x,A,tau):
    return A -tau*x

def residuals_simple(params, x, y):
    A1,tau1 = params
    return y - exp_simple_func(x, A1, tau1)

def residuals_double(params, x, y):
    A1,A2,tau1,tau2 = params
    return y - exp_double_func(x, A1, A2,tau1,tau2)


In [ ]:
init=450
init_simple=init+250
width=300

fit_double=[np.zeros(4) for _ in range(n_run)]
fit_simple=[np.zeros(2) for _ in range(n_run)]

for n in range(n_run):
    y_data=(scint_avg[n])[init:init+width]
    x_data=np.arange(0,len(y_data))

    y_data_2=(scint_avg[n])[init_simple:init+width]
    x_data_2=np.arange(0,len(y_data_2))

    initial_guess = [1.5,1.5,100,1]
    initial_guess_2 = [1.5,100]
    result = least_squares(residuals_double, initial_guess, args=(x_data, y_data),method='dogbox')
    result_2 = least_squares(residuals_simple, initial_guess_2, args=(x_data_2, y_data_2),method='dogbox')
    fit_double[n]=result.x
    fit_simple[n]=result_2.x

In [ ]:
my_index=7
plt.plot((scint_avg[my_index])[init:init+width])
plt.plot(exp_double_func(x_data,*fit_double[my_index]))
plt.yscale("log")

In [ ]:
y_slow = [value[2] for value in fit_double]
y_slow_2 = [value[1] for value in fit_simple]

plt.plot(np.divide(x,360),np.multiply(y_slow,16),".")
plt.plot(np.divide(x,360),np.multiply(y_slow_2,16),".",color="red")
plt.xlabel("Efield [kV/cm]")
plt.ylabel("Slow component [ns]")

In [ ]:
y_fast = [value[3] for value in fit_double]

plt.plot(np.divide(x,360),np.multiply(y_fast,16),".")
plt.xlabel("Efield [kV/cm]")
plt.ylabel("Fast component [ns]")

In [ ]:
np.zeros(0)